# The per-speaker error map — committees corpus

**Toward Personalized Hebrew ASR** · Dolev Abudi, Hadas Yonat

Stage 1's analysis (`speaker_error_map.ipynb`, run on VoxKnesset) re-done on the corpus the
project moved to. Same two questions:

1. How well does each model do **per speaker**?
2. What is each speaker's **adaptation gain** — how much does the Hebrew fine-tune help them?

| | model | inference |
|---|---|---|
| **A** | `openai/whisper-large-v3` (general), language forced to Hebrew | `Dolevabudi/knesset-committees-inference`, `hypothesis_A` |
| **B** | `ivrit-ai/whisper-large-v3-turbo-ct2` (Hebrew fine-tune) | same, `hypothesis_B` |

The data is the Stage-1 subset: one hour per MK (alignment quality ≥ 0.5), drawn round-robin
over sessions — 65,990 chunks of ≤ 30 s, 230 h, 267 speakers, every chunk transcribed by both
arms and validated end to end (`docs/inference.md` § Result).

Every function used here lives in `src/evaluation/error_map.py`; this notebook is the narrative
and the figures. `python src/evaluation/error_map.py --run` writes the same tables without it.

---
## 1. Load and score

In [ ]:
import os, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.ticker as mticker
from scipy import stats
sys.path.insert(0, os.path.join('..', 'src', 'evaluation'))
import error_map as E

C_A, C_B, C_GAIN = '#0072B2', '#D55E00', '#009E73'      # model A blue, B orange, gain green
FIG = os.path.join('..', 'docs', 'figures', 'error_map'); os.makedirs(FIG, exist_ok=True)
def save(name): plt.savefig(os.path.join(FIG, name + '.png'), dpi=130, bbox_inches='tight')
pd.set_option('display.width', 160)

seg = E.count_errors(E.load_chunks())
print(f'{len(seg):,} chunks, {seg.speaker_id.nunique()} speakers, {seg.duration_s.sum()/3600:.1f} hours; '
      f'both arms scored with evaluate.score (Stage 1 normalisation, word/char Levenshtein, S/D/I)')
seg[['chunk_id', 'speaker_id', 'duration_s', 'quality', 'n_words', 'werr_A', 'werr_B', 'wpm']].head(3)

---
## 2. The one filter: alignment quality

Stage 1 dropped segments whose reference could not account for their audio — a 4.2-hour file
with a 126-word reference. Its rule (a flat 300 s of unexplained audio) cannot fire on ≤ 30 s
chunks. But the failure it caught, *reference text that does not match the audio*, is exactly
what this corpus's per-chunk alignment `quality` measures: the median probability the aligner
assigned to the chunk's words. The table below is why the threshold is 0.7 — the value
`docs/committees_handoff.md` recommends to any consumer.

In [ ]:
E.by_bucket(seg, 'quality', [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]).round(3)

Below 0.7 the WER sits near 1.0 under **both** arms: the models transcribe speech the reference
does not contain, and every such word is an insertion. Above 0.9 — 60 % of the chunks, 74 % of
the hours — the corpus looks like a corpus. The filter keeps quality ≥ 0.7 and reports what it
does, as Stage 1 did: the corpus before, after, and the dropped chunks on their own.

In [ ]:
kept, dropped, effect = E.qc(seg)
print(f'dropped {len(dropped):,} of {len(seg):,} chunks ({len(dropped)/len(seg):.1%}), '
      f'{dropped.duration_s.sum()/3600:.1f} h;  speakers before {seg.speaker_id.nunique()}, after {kept.speaker_id.nunique()}')
for tag in ('A', 'B'):
    print(f'model {tag}:  {len(dropped)/len(seg):.1%} of the chunks and {dropped.n_words.sum()/seg.n_words.sum():.1%} of the '
          f'reference words carry {dropped[f"werr_{tag}"].sum()/seg[f"werr_{tag}"].sum():.1%} of all word errors')
effect.round(4)

**Reading it.** Unlike VoxKnesset, where fourteen segments could not move the average, here the
filter moves the corpus WER by three points on both arms (A 0.417 → 0.387, B 0.324 → 0.292):
12 % of the chunks hold 5 % of the reference words and 12–14 % of the errors. The dropped
chunks score 0.99 and 0.92 — errors equal to reference words, which is what a reference that
does not match its audio looks like.

The filter is applied uniformly, and both arms pay it equally, so the *difference* between
arms — the quantity every later section is about — barely changes (0.093 before, 0.094 after).

---
## 3. Overall, before looking at speakers

In [ ]:
for tag, name in [('A', 'whisper-large-v3 (general, language forced)'), ('B', 'ivrit-ai whisper-large-v3-turbo-ct2 (fine-tune)')]:
    print(f'{name:<50}  WER {kept[f"werr_{tag}"].sum()/kept.n_words.sum():.4f}   CER {kept[f"cerr_{tag}"].sum()/kept.n_chars.sum():.4f}   '
          f'errors: S {kept[f"S_{tag}"].sum()/kept[f"werr_{tag}"].sum():.0%}  D {kept[f"D_{tag}"].sum()/kept[f"werr_{tag}"].sum():.0%}  '
          f'I {kept[f"I_{tag}"].sum()/kept[f"werr_{tag}"].sum():.0%}   runaway chunks {int(kept[f"runaway_{tag}"].sum())}')

Two things differ from the plenums. The level is roughly double (Stage 1: A 0.207, B 0.098):
the reference is a cleaned stenographic protocol while both models transcribe the repetitions,
false starts and cross-talk of a committee room. And the error *mix* differs between arms: over
half of B's errors are insertions against 38 % for A. B transcribes more of what the protocol
left out — its errors are the register's, not the speaker's — which is the same reason the
comparison below is a *gain* and not an absolute WER.

---
## 4. Performance per speaker

The same calculation grouped by speaker, with what Stage 1 left for later: a 95 % bootstrap CI
per speaker (chunks resampled, Σerrors / Σwords per resample), so the ranking's noise is visible.

In [ ]:
spk = E.per_speaker(kept, n_boot=1000)
print(f'{len(spk)} speakers; {int(spk.reliable.sum())} with >= {E.MIN_SEG} chunks')
print(f'median CI half-width: WER_B ±{((spk.wer_B_hi - spk.wer_B_lo)/2).median():.3f}, gain ±{((spk.gain_abs_hi - spk.gain_abs_lo)/2).median():.3f}')
spk[['n_seg', 'hours', 'wer_A', 'wer_B', 'cer_A', 'cer_B']].describe().round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
bins = np.linspace(0, spk.wer_A.max(), 45)
ax.hist(spk.wer_A, bins=bins, color=C_A, alpha=0.8, label='A — general')
ax.hist(spk.wer_B, bins=bins, color=C_B, alpha=0.8, label='B — Hebrew fine-tune')
ax.set_xlabel('WER of one speaker'); ax.set_ylabel('number of speakers')
ax.set_title('Performance per speaker'); ax.legend(); save('per_speaker_hist'); plt.show()

---
## 5. Adaptation gain per speaker

* `gain_abs` = WER_A − WER_B — WER points the fine-tune recovers for that speaker
* `gain_rel` = gain_abs / WER_A — the fraction of the general model's error it removes

In [ ]:
print(f'speakers helped by the fine-tune : {(spk.gain_abs > 0).sum()} / {len(spk)}')
print(f'speakers hurt by it              : {(spk.gain_abs < 0).sum()} / {len(spk)}')
print(f'gain CI excludes zero for        : {(spk.gain_abs_lo > 0).sum()} / {len(spk)}')
print(f'median relative gain             : {spk.gain_rel.median():.1%}   (Stage 1 on the plenums: 51.7%)')
print(f'rank correlation, WER_A vs WER_B : {stats.spearmanr(spk.wer_A, spk.wer_B).statistic:+.3f}')
print(f'rank correlation, WER_A vs gain  : abs {stats.spearmanr(spk.wer_A, spk.gain_abs).statistic:+.3f}, rel {stats.spearmanr(spk.wer_A, spk.gain_rel).statistic:+.3f}')
spk[['wer_A', 'wer_B', 'gain_abs', 'gain_rel']].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ax = axes[0]; lim = [0, spk.wer_A.max() * 1.05]
ax.plot(lim, lim, 'k--', lw=1, label='no change')
ax.errorbar(spk.wer_A, spk.wer_B, yerr=[spk.wer_B - spk.wer_B_lo, spk.wer_B_hi - spk.wer_B], fmt='none', ecolor=C_GAIN, alpha=0.25, lw=0.8)
ax.scatter(spk.wer_A, spk.wer_B, s=18, color=C_GAIN, alpha=0.7, edgecolor='white')
ax.set_xlim(lim); ax.set_ylim(lim); ax.set_xlabel('WER — model A'); ax.set_ylabel('WER — model B')
ax.set_title('Every point is one speaker (bar: 95% CI on B)'); ax.legend()
ax = axes[1]
ax.hist(spk.gain_rel, bins=40, color=C_GAIN, alpha=0.85)
ax.set_xlabel('relative gain (fraction of error removed)'); ax.set_ylabel('number of speakers'); ax.set_title('Adaptation gain per speaker')
plt.tight_layout(); save('gain'); plt.show()

### Who gains most, and who gains least

Reliable speakers only (≥ 20 chunks). The CI columns are the point: two speakers whose gains
differ by less than their intervals are not in a meaningful order.

In [ ]:
cols = ['n_seg', 'hours', 'wer_A', 'wer_B', 'gain_abs', 'gain_abs_lo', 'gain_abs_hi', 'gain_rel']
r = spk[spk.reliable]
print('--- largest relative gain ---'); display(r.nlargest(10, 'gain_rel')[cols].round(3))
print('--- smallest relative gain ---'); display(r.nsmallest(10, 'gain_rel')[cols].round(3))

---
## 6. WER against the amount of data per speaker

Stage 1 asked whether prolific speakers are recognised differently (no) and used the plot to show
how much of the per-speaker spread is measurement noise (a lot, below 20 segments). Two things
change here.

The subset caps every speaker at one hour, so `hours` in the subset is not volume — it is what
survived the quality filter. The volume axis is therefore **corpus hours**: all of the speaker's
chunks in `Hadasy/knesset-committees-chunks` at quality ≥ 0.7, which is also what a subgroup or
personal fine-tune could train on.

And the filter introduces a confound worth stating: the share of a speaker's chunks the filter
removed correlates strongly with their WER on the chunks that remain. Speakers whose speech
aligns badly are also harder to recognise — cross-talk, or labels the protocol got wrong. That is
the handoff's warning (*suspect the labels before the model*) as a number.

In [ ]:
attrs = E.speaker_attrs(kept)
totals = kept.groupby('speaker_id').agg(werr_A_total=('werr_A', 'sum'), werr_B_total=('werr_B', 'sum'))
G = spk.join(totals).join(attrs)
G['qc_dropped'] = 1 - kept.groupby('speaker_id').size() / seg.groupby('speaker_id').size()

def rho(x, y):
    r = stats.spearmanr(x, y); return r.statistic, r.pvalue
for col in ('hours_corpus', 'n_seg', 'qc_dropped'):
    print(f'{col:<13}', '   '.join(f'vs wer_{t}: rho {rho(G[col], G[f"wer_{t}"])[0]:+.3f} (p {rho(G[col], G[f"wer_{t}"])[1]:.2g})' for t in 'AB'))

def decimal_log_axis(ax, ticks):
    ax.set_xscale('log'); ax.xaxis.set_major_locator(mticker.FixedLocator(ticks)); ax.xaxis.set_minor_locator(mticker.NullLocator())
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:g}'))

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), sharey=True)
for ax, col, label, ticks, cut in [(axes[0], 'n_seg', 'chunks per speaker (in the subset)', [5, 10, 30, 100, 300, 500], 20),
                                    (axes[1], 'hours_corpus', 'hours of speech in the corpus (quality ≥ 0.7)', [0.1, 0.3, 1, 3, 10, 30, 100, 300], None)]:
    for tag, colour, name in [('A', C_A, 'A — general'), ('B', C_B, 'B — Hebrew fine-tune')]:
        r_, p_ = rho(G[col], G[f'wer_{tag}'])
        ax.scatter(G[col].clip(lower=0.05), G[f'wer_{tag}'], s=18, color=colour, alpha=0.55, edgecolor='white', linewidth=0.6,
                   label=f'{name}   rho = {r_:+.3f} (p = {p_:.2g})')
    if cut is not None:
        ax.axvline(cut, color='0.4', ls=':', lw=1); ax.text(cut, 0.02, f' {cut} chunks', transform=ax.get_xaxis_transform(), fontsize=7.5, color='0.4', va='bottom')
    decimal_log_axis(ax, ticks); ax.set_yscale('log')
    ax.yaxis.set_major_locator(mticker.FixedLocator([0.1, 0.2, 0.3, 0.5, 0.8])); ax.yaxis.set_minor_locator(mticker.NullLocator())
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:g}'))
    ax.set_xlabel(label); ax.legend(fontsize=8, loc='upper right'); ax.grid(alpha=0.3)
axes[0].set_ylabel('WER of one speaker')
fig.suptitle('WER against how much speech we have from each speaker', fontsize=12, fontweight='bold')
plt.tight_layout(); save('volume'); plt.show()

In [ ]:
bucket = pd.cut(G.hours_corpus, [0, 1, 3, 10, 30, 100, 1000], labels=['<1 h', '1-3 h', '3-10 h', '10-30 h', '30-100 h', '100+ h'])
display(G.groupby(bucket, observed=True).agg(speakers=('wer_A', 'size'), median_wer_A=('wer_A', 'median'), sd_wer_A=('wer_A', 'std'),
                                             median_wer_B=('wer_B', 'median'), sd_wer_B=('wer_B', 'std')).round(4))
display(E.volume_table(spk).round(4))

### Reading it

**The level is flat.** Corpus hours do not predict either model's WER (rho within ±0.07). A
speaker with 100 hours of committee speech is recognised no better than one with two.

**The spread is a measurement effect, again.** The standard deviation of WER falls from about
0.15–0.21 among speakers with under 20 chunks or under an hour to about 0.06–0.08 in the large
buckets, as it did on VoxKnesset. Six speakers have fewer than 20 chunks after the filter; they
are kept in the tables and excluded from every statistical claim below.

**The one correlation that is real is with the filter.** The share of a speaker's chunks the
quality filter removed predicts their WER on the kept chunks with rho ≈ 0.6–0.7. Nothing here
can say whether that is acoustics (cross-talk, poor microphones) or labels (the protocol naming
the wrong speaker); the audio gate in `speaker_index/validate_audio.py` is the check, and it has
not run. Until it does, an anomalous speaker is a labelling question first.

---
## 7. Subgroups — who could share a fine-tune?

Stage 2's sharing axis trains **personal-only**, **subgroup-only** and **subgroup-then-personal**
adapters, and needs a subgroup *definition*. A definition has to clear two independent bars:
enough data to train on (hours from the *other* members, since the target is held out), and
actual separation — members resembling each other in how the models treat them. The rules are
Stage 1's, unchanged: demographic (gender, nationality, religion, religious orientation, country
of origin, age quartile) and behavioural (speaking-rate tertile). Demographics come from the
speaker index; hours are corpus hours at quality ≥ 0.7.

In [ ]:
for name, col in E.RULES.items():
    counts = G[col].value_counts()
    print(f'{name:<24} {counts.size} groups   ' + ', '.join(f'{k}={v}' for k, v in counts.items()))

### 7.1 Is there enough data to train on?

`hours_minus_largest` is the corpus speech left in a subgroup once its biggest speaker is held
out. Thresholds as in Stage 1: at least 3 speakers and 5 hours.

In [ ]:
for name, col in E.RULES.items():
    print(f'\n===== {name} =====')
    display(E.subgroup_table(G, col).round({'hours_subset': 1, 'hours_corpus': 1, 'hours_minus_largest': 1, 'wer_A': 4, 'wer_B': 4, 'gain_abs': 4, 'gain_rel': 3}))

### 7.2 Does the rule actually separate the speakers?

One dot per reliable speaker, each subgroup in its own colour; bar = group median, line = IQR,
dashed = corpus median. `Unknown` and `Unrecorded` are missing metadata, not groups, and are
dropped here (not from the viability table).

In [ ]:
PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9', '#8B4513', '#555555']
print(f'separation analysis uses speakers with >= {E.MIN_SEG} chunks, in groups of >= {E.MIN_SPEAKERS}, excluding {sorted(E.NOT_A_GROUP)}\n')
for name, col in E.RULES.items():
    d = E.separation_set(G, col); print(f'  {name:<24} {len(d):>3} of {len(G)} speakers, {d[col].nunique()} groups')

def separation_plot(name, col):
    d = E.separation_set(G, col)
    order = d.groupby(col, observed=True).gain_rel.median().sort_values().index.tolist()[:len(PALETTE)]
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4)); rng_j = np.random.default_rng(0)
    for ax, ycol, ylab in [(axes[0], 'gain_rel', 'relative gain (fraction of error removed)'), (axes[1], 'gain_abs', 'absolute gain (WER points)')]:
        ax.axhline(d[ycol].median(), color='0.55', ls='--', lw=1, zorder=1)
        for k, gname in enumerate(order):
            v = d.loc[d[col] == gname, ycol].dropna().values
            if not len(v): continue
            x = k + rng_j.uniform(-0.20, 0.20, len(v))
            ax.scatter(x, v, s=26, color=PALETTE[k], alpha=0.70, edgecolor='white', linewidth=0.7, zorder=3)
            q1, med, q3 = np.percentile(v, [25, 50, 75])
            ax.plot([k, k], [q1, q3], color=PALETTE[k], lw=2.5, alpha=0.55, zorder=2); ax.plot([k - 0.30, k + 0.30], [med, med], color=PALETTE[k], lw=3, zorder=4)
        ax.set_xticks(range(len(order))); ax.set_xticklabels([f'{g}\nn={(d[col] == g).sum()}' for g in order], fontsize=8)
        ax.set_ylabel(ylab); ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True); ax.set_xlim(-0.6, len(order) - 0.4)
    fig.suptitle(f'Subgroup rule: {name}', fontsize=12, fontweight='bold'); plt.tight_layout(); save('separation_' + col); plt.show()

for name, col in E.RULES.items():
    separation_plot(name, col)

### 7.3 Ranking the rules

**Eta squared** is the share of the variance in a speaker-level outcome that sits *between*
subgroups. It is biased upward for small, numerous groups, so each rule is compared with its own
permutation null (labels shuffled across speakers, 2,000 times, 95th percentile); a rule below its
null is noise dressed as structure.

In [ ]:
rank = E.rank_rules(G, n_perm=2000)
rank.round(4)

### 7.4 Reading it

**Every genuine group is viable**, as on the plenums: the smallest — Bedouin (5 speakers, 27 h),
Europe (5, 68 h), Christian (3, 76 h) — all clear 5 hours with their largest member held out.
Only the metadata-missing cells fail. The question is again whether a rule separates anything.

**The two outcomes come apart — but the other way round from VoxKnesset.** There, demographic
membership predicted who the models find *difficult* (`wer_B`) and nothing but speaking rate
predicted who *benefits*. Here:

| separates `gain_rel` (who benefits from adaptation) | separates `wer_B` (who is hard) |
|---|---|
| speaking rate (eta² 0.09, the strongest), religion, nationality, age quartile, gender at the edge | religious orientation (eta² 0.06), gender |
| country of origin: no (p = 0.09) | speaking rate, religion, nationality, age, origin: no |

Speaking rate is the one rule that separated gain on both corpora, and it is the strongest here:
slow speakers gain most (27 % of A's error removed, median) and fast speakers least (22 %). Arab,
Muslim and Bedouin speakers gain more than Jewish speakers (29–30 % against 24 %) with no
difference in how hard they are for B — the fine-tune closes a gap the general model has on
them. Religious orientation is the strongest difficulty rule: Haredi speakers score highest under
both arms (B 0.334 against 0.285 for secular) and gain no more than anyone else.

**Keep the effect sizes in view.** The best rule explains 9 % of the between-speaker variance in
gain and 6 % in `wer_B`; 90 % of the variation is *within* every group. Real, weak, and — the
useful part for Stage 2 — a different set of rules than the plenums produced, which says the
plenum result was partly a property of that corpus and not of the speakers.

---
## 8. Who to adapt

Design step 5: the speakers the generic Hebrew fine-tune helped **least** are where per-speaker
adaptation has room to act. Lowest relative gain first, reliable speakers only, with the CI so the
order's noise is visible. Whether a candidate's low gain is a property of the voice or of the
labels is the question §6 leaves open.

In [ ]:
cand = E.candidates(spk).join(attrs[['speaker_name', 'gender', 'origin', 'orientation', 'rate_band', 'hours_corpus']])
med = spk[spk.reliable].gain_abs.median()
print(f'{int((cand.gain_abs_hi < med).sum())} speakers whose gain is below the median ({med:.3f} WER points) with the whole CI')
cand.head(20).round(3)

### 8.1 The panel

The eleven speakers chosen for adaptation (`src/evaluation/outputs/committees_panel.csv`,
reasons in `docs/adaptation_plan.md` § The panel). Four views: the numbers with their CIs;
where they sit on the map of all 267; what kind of errors each model makes on them; and their
WER session by session, with the sessions held back as personal-test marked — the point being
that the ranking used only the earlier sessions, so the held-out ones are an independent check
that a speaker's profile is real and not one bad afternoon.

In [ ]:
panel = pd.read_csv(os.path.join('..', 'src', 'evaluation', 'outputs', 'committees_panel.csv'), index_col=0)
core = panel[~panel.profile.str.contains('alt')]
ids = core.index.tolist()
view = core[['profile', 'speaker_name', 'gender', 'rate_band', 'orientation']].join(
    spk.loc[ids, ['n_seg', 'wer_A', 'wer_A_lo', 'wer_A_hi', 'wer_B', 'wer_B_lo', 'wer_B_hi', 'gain_abs', 'gain_abs_lo', 'gain_abs_hi', 'gain_rel']])
view.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
ax = axes[0]; lim = [0.15, spk.wer_A.max() * 1.03]
ax.plot(lim, lim, 'k--', lw=1, label='no change')
ax.scatter(spk.wer_A, spk.wer_B, s=14, color='0.75', alpha=0.6, edgecolor='white', label='all 267 speakers')
colors = {'S1': '#D55E00', 'S2': '#CC79A7', 'S3': '#0072B2', 'S4': '#009E73', 'C': '#E69F00'}
for sid, r in core.iterrows():
    s = spk.loc[sid]; c = colors[r.profile]
    ax.errorbar(s.wer_A, s.wer_B, xerr=[[s.wer_A - s.wer_A_lo], [s.wer_A_hi - s.wer_A]], yerr=[[s.wer_B - s.wer_B_lo], [s.wer_B_hi - s.wer_B]], fmt='o', color=c, ms=7, capsize=2, lw=1)
    ax.annotate(f'{r.speaker_name}', (s.wer_A, s.wer_B), xytext=(5, 4), textcoords='offset points', fontsize=7.5, color=c)
for k, c in colors.items(): ax.scatter([], [], color=c, s=40, label={'S1': 'S1  fine-tune failed them', 'S2': 'S2  simply hard', 'S3': 'S3  typical, left behind', 'S4': 'S4  headroom', 'C': 'C   control, high gain'}[k])
ax.set_xlim(lim); ax.set_ylim(0.1, 0.6); ax.set_xlabel('WER — model A'); ax.set_ylabel('WER — model B'); ax.legend(fontsize=7.5, loc='upper left'); ax.set_title('The panel on the map (bars: 95% CI)')

ax = axes[1]
order = core.sort_values(['profile', 'speaker_name']).index
x = np.arange(len(order)); w = 0.38
sa, sb = spk.loc[order], spk.loc[order]
ax.bar(x - w/2, sa.wer_A, w, color=C_A, alpha=0.85, label='A — general', yerr=[sa.wer_A - sa.wer_A_lo, sa.wer_A_hi - sa.wer_A], capsize=2)
ax.bar(x + w/2, sb.wer_B, w, color=C_B, alpha=0.85, label='B — Hebrew fine-tune', yerr=[sb.wer_B - sb.wer_B_lo, sb.wer_B_hi - sb.wer_B], capsize=2)
ax.axhline(spk.wer_B.median(), color=C_B, ls=':', lw=1); ax.axhline(spk.wer_A.median(), color=C_A, ls=':', lw=1)
ax.set_xticks(x); ax.set_xticklabels([f'{core.loc[s, "profile"]}\n{core.loc[s, "speaker_name"]}' for s in order], fontsize=7, rotation=60, ha='right')
ax.set_ylabel('WER'); ax.set_title('Per speaker (dotted: corpus medians)'); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); save('panel_map'); plt.show()

In [ ]:
# error mix: what each model gets wrong on each panel speaker
fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.8), sharey=True)
for ax, tag, name in [(axes[0], 'A', 'A — general'), (axes[1], 'B', 'B — Hebrew fine-tune')]:
    d = spk.loc[order, [f'S_share_{tag}', f'D_share_{tag}', f'I_share_{tag}']]
    bottom = np.zeros(len(order))
    for col, lab, c in [(f'S_share_{tag}', 'substitutions', '#0072B2'), (f'D_share_{tag}', 'deletions', '#56B4E9'), (f'I_share_{tag}', 'insertions', '#D55E00')]:
        ax.bar(x, d[col], 0.7, bottom=bottom, color=c, label=lab); bottom += d[col].values
    ax.set_xticks(x); ax.set_xticklabels([core.loc[s, 'speaker_name'] for s in order], fontsize=7, rotation=60, ha='right')
    ax.set_title(f'{name}: share of word errors by type'); ax.grid(axis='y', alpha=0.3)
axes[0].set_ylabel('share of word errors'); axes[1].legend(fontsize=8, loc='lower right')
plt.tight_layout(); save('panel_error_mix'); plt.show()

In [ ]:
# WER by session over time; the latest ~35% of words per speaker (held out as personal-test) marked
def session_wer(sid):
    d = kept[kept.speaker_id == sid]
    per = d.groupby('session').agg(date=('session_date', 'first'), words=('n_words', 'sum'), eA=('werr_A', 'sum'), eB=('werr_B', 'sum'), chunks=('chunk_id', 'size')).sort_values('date')
    per['wer_A'] = per.eA / per.words; per['wer_B'] = per.eB / per.words
    cum = per.words[::-1].cumsum()[::-1] / per.words.sum()          # share of words from this session to the end
    per['test_side'] = cum <= 0.35
    return per
fig, axes = plt.subplots(3, 4, figsize=(15, 8.5), sharey=True); axes = axes.ravel()
for k, sid in enumerate(order):
    per = session_wer(sid); ax = axes[k]; r = core.loc[sid]
    t = pd.to_datetime(per.date)
    ax.scatter(t[~per.test_side], per.wer_B[~per.test_side], s=per.chunks[~per.test_side] * 3, color=C_B, alpha=0.65, label='B, train-side sessions')
    ax.scatter(t[per.test_side], per.wer_B[per.test_side], s=per.chunks[per.test_side] * 3, color=C_B, alpha=0.65, marker='s', edgecolor='k', label='B, held-out (personal-test)')
    ax.scatter(t[~per.test_side], per.wer_A[~per.test_side], s=per.chunks[~per.test_side] * 3, color=C_A, alpha=0.45, label='A, train-side')
    ax.scatter(t[per.test_side], per.wer_A[per.test_side], s=per.chunks[per.test_side] * 3, color=C_A, alpha=0.45, marker='s', edgecolor='k', label='A, held-out')
    ax.axhline(spk.loc[sid, 'wer_B'], color=C_B, lw=1, ls=':'); ax.axhline(spk.loc[sid, 'wer_A'], color=C_A, lw=1, ls=':')
    tr_, te_ = per[~per.test_side], per[per.test_side]
    ax.set_title(f'{r.profile}  {r.speaker_name}   B: train {tr_.eB.sum()/tr_.words.sum():.2f} | held-out {te_.eB.sum()/te_.words.sum():.2f}', fontsize=8.5)
    ax.tick_params(axis='x', labelsize=7, rotation=30); ax.set_ylim(0, 1.0); ax.grid(alpha=0.25)
axes[-1].axis('off'); axes[0].set_ylabel('WER (one dot per session, size = chunks)'); axes[4].set_ylabel('WER'); axes[8].set_ylabel('WER')
h, l = axes[0].get_legend_handles_labels(); axes[-1].legend(h, l, loc='center', fontsize=9)
fig.suptitle('Panel speakers, WER by session over time; squares are the sessions held back as personal-test', fontsize=12, fontweight='bold')
plt.tight_layout(); save('panel_sessions'); plt.show()

**Reading it.** A speaker belongs on the panel if the held-out sessions tell the same story as
the sessions the choice was made on: a train-side and held-out WER_B within a few points of
each other, and the dots clustered around the dotted line rather than split into two eras. A
speaker whose held-out sessions sit far from the train-side ones has a profile that is
partly time or session, not voice, and should be swapped for an alternate before GPU time.

---
## 9. Two things to distrust

**Sessions where both arms fail.** The handoff warns that a time-origin offset between audio and
alignment would show as a whole session near WER 1.0 under *both* models. These sessions are
listed for a listening check, not removed: their chunks passed the quality filter.

In [ ]:
flagged = E.flagged_sessions(kept)
print(f'{len(flagged)} sessions with >= 5 chunks and WER > 0.9 under both arms, {int(flagged.chunks.sum())} chunks')
flagged.round(3)

**Language detection, the reason Arm A is told the language.** A's first run let Whisper detect
the language. By chunk length, the share of hypotheses that came back in another script:

In [ ]:
lang, share = E.language_detection(seg)
print(f'{share:.1%} of all chunks; by duration:'); lang.round(3)

---
## 10. Save

In [ ]:
R = E.run(verbose=False)
print('wrote', sorted(f for f in os.listdir(E.OUT) if f.startswith('committees_')))
R['spk'][['n_seg', 'wer_A', 'wer_B', 'gain_abs', 'gain_rel']].head()

---
## 11. Beyond Stage 1 — what the inference also tells

Stage 1's questions are answered above. The committees corpus raises others, and the inference
can answer them without any new run. Functions in `src/evaluation/error_analysis.py`; every
table is also written by `python src/evaluation/error_analysis.py --run`.

In [ ]:
import error_analysis as X
k = X.with_committee(kept)

### 11.1 Conditions: when and where is it hard?

Both arms, by year and by committee room. If these move, they move on top of the speaker —
which is why the splits are by date and why a panel speaker's held-out sessions were checked.

In [ ]:
display(X.by_condition(k, 'year').round(3))
display(X.by_condition(k, 'committee_name', top=12).round(3))
fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.8))
t = X.by_condition(k, 'year'); ax = axes[0]
ax.plot(t.index, t.WER_A, 'o-', color=C_A, label='A — general'); ax.plot(t.index, t.WER_B, 'o-', color=C_B, label='B — Hebrew fine-tune')
ax.set_ylabel('corpus WER'); ax.set_title('By year of the session'); ax.legend(); ax.grid(alpha=0.3)
t = X.by_condition(k, 'committee_name', top=12); ax = axes[1]; y = np.arange(len(t))
ax.barh(y - 0.2, t.WER_A, 0.4, color=C_A, label='A'); ax.barh(y + 0.2, t.WER_B, 0.4, color=C_B, label='B')
ax.set_yticks(y); ax.set_yticklabels([f'{n[:28]}  (n={int(c)})' for n, c in zip(t.index, t.chunks)], fontsize=7.5); ax.invert_yaxis()
ax.set_xlabel('corpus WER'); ax.set_title('By committee (12 largest)'); ax.legend(fontsize=8); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); save('conditions'); plt.show()

### 11.2 What the errors are

Stage 1 listed the digit problem as an open issue. Here the substitutions are counted: which
pairs recur, how many are one character apart (orthography, a dropped conjunction), how many
involve a digit, and what WER is with numeric tokens removed from both sides.

In [ ]:
content = X.error_content(kept)
for tag in 'AB':
    c = content[tag]
    print(f"{tag}: near-miss substitutions {c['near_miss_share_of_substitutions']:.1%} of substitutions = {c['near_miss_share_of_errors']:.1%} of errors | "
          f"errors involving a digit {c['digit_share_of_errors']:.1%} | WER {c['wer']:.4f} -> {c['wer_without_numbers']:.4f} without numeric tokens")
    print('   top substitutions: ' + ', '.join(f'{a}→{b} ({n})' for (a, b), n in c['top_substitutions'][:10]))
    print('   top insertions:    ' + ', '.join(f'{w} ({n})' for w, n in c['top_insertions'][:10]))
    print('   top deletions:     ' + ', '.join(f'{w} ({n})' for w, n in c['top_deletions'][:10]))

### 11.3 Are B's insertions real speech?

Over half of B's errors are insertions. If B were hallucinating, its inserted words would be its
own; if the protocol simply left them out, the other model would hear them too. The share of
B's inserted words that also appear in A's hypothesis of the same chunk is that test. Looping
and runaway decodes are the hallucination rates proper.

In [ ]:
print(X.insertion_agreement(kept))
display(X.hallucination(kept).round(4))
print(X.chunk_level(kept))

### 11.4 Is the per-speaker ranking reliable?

Split each speaker's sessions into odd and even halves, compute the measure on each, correlate
across speakers, and correct to full length (Spearman-Brown). A reliability near 1 means the
spread between speakers is real; near 0 means it is one hour's sampling noise.

In [ ]:
rel, H = X.split_half(kept); display(rel.round(3))
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))
for ax, m, c, name in [(axes[0], 'wer_B', C_B, 'WER_B'), (axes[1], 'gain_rel', C_GAIN, 'relative gain')]:
    ax.scatter(H[f'{m}_1'], H[f'{m}_2'], s=16, color=c, alpha=0.6, edgecolor='white'); lim = [min(H[f'{m}_1'].min(), H[f'{m}_2'].min()), max(H[f'{m}_1'].max(), H[f'{m}_2'].max())]
    ax.plot(lim, lim, 'k--', lw=1); ax.set_xlabel(f'{name}, odd sessions'); ax.set_ylabel(f'{name}, even sessions'); ax.set_title(f'{name}: reliability {rel.loc[m, "reliability"]:.2f}'); ax.grid(alpha=0.3)
plt.tight_layout(); save('reliability'); plt.show()

### 11.5 Forcing the language: what it did to A

Arm A's first run let Whisper detect the language. On the same chunks, the auto-detect and the
forced-Hebrew WER by chunk length.

In [ ]:
lang, wa_auto, wa = X.language_forcing(kept)
print(f'corpus WER A: auto-detect {wa_auto:.4f}, forced Hebrew {wa:.4f}'); lang.round(3)

### 11.6 The same speakers on the plenums

Stage 1 measured most of these speakers on VoxKnesset. Does difficulty transfer across corpora,
and how far apart are the levels? The plenum number was measured on audio arm B had trained on,
in the cleaner register.

In [ ]:
cc, J = X.cross_corpus(spk); display(cc.round(3))
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))
for ax, m, c, name in [(axes[0], 'wer_B', C_B, 'WER_B'), (axes[1], 'gain_rel', C_GAIN, 'relative gain')]:
    ax.scatter(J[f'plenum_{m}'], J[m], s=16, color=c, alpha=0.6, edgecolor='white'); ax.set_xlabel(f'{name} on the plenums (Stage 1)'); ax.set_ylabel(f'{name} on the committees')
    ax.set_title(f'{name}: Spearman {cc.loc[m, "spearman"]:+.2f}, {int(cc.loc[m, "speakers"])} speakers'); ax.grid(alpha=0.3)
plt.tight_layout(); save('cross_corpus'); plt.show()

### 11.7 Reading it

**The residual is register first, voice second.** Three quarters of B's inserted words are
words A also heard: the protocol did not write them down. The substitutions that recur are a
dropped conjunction, full versus defective spelling, and a stenographer's synonym. None of
this is what a personal adapter should learn, and all of it will be counted as improvement if
an adapter learns it — the S/D/I split in Stage 2 is not optional.

**The digit problem is 1 % of errors.** It can be left where it is.

**The ranking is real.** With reliability of 0.75–0.85, the per-speaker spread and the
candidates list are measured properties, not one hour's luck; the subgroup rules are weak on
a reliable outcome.

**Time and room matter**, several points each, which is what the date-ordered split and the
held-out check on the panel are for.

**The plenum numbers were a different measurement.** Difficulty transfers between corpora
(rho 0.4–0.5); the level does not (3.2× per speaker).

In [ ]:
X.run()      # committees_conditions.csv, committees_error_content.csv, committees_checks.json

---
## 12. The same map, protocol-aware

§11 showed that over half of B's errors are insertions and that three quarters of B's inserted
words also appear in A's hypothesis of the same chunk: speech the stenographer did not write
down. Charging both models for the protocol's tidying measures the protocol, not recognition.

This section re-runs the whole map under one alternative count, **forgiven-shared WER**: an
inserted word that the *other* model also produced at that chunk is not charged. Substitutions
and deletions are unchanged, the denominator is unchanged, and an inserted word only one model
produces is still charged. Standard WER stays the headline (it is what Stage 1 and the
literature report); this is the second column.

The caveat, stated once and meant everywhere: A and B are both Whisper large-v3 descendants,
so a shared insertion is strong evidence of a protocol omission, not proof. Two correlated
models can hallucinate alike. The way to settle it is a small human verbatim transcription,
which this version defers.

In [ ]:
kf = X.forgiven_counts(kept)
rows = {}
for name, f in (('standard', ''), ('forgiven-shared', '_f')):
    c = E.corpus_scores(kf, f); rows[name] = dict(WER_A=c.WER_A, WER_B=c.WER_B, B_advantage=(c.WER_A - c.WER_B) / c.WER_A)
pd.DataFrame(rows).T.round(4)

In [ ]:
spk_f = E.per_speaker(kf, n_boot=1000, suffix='_f')
fig, axes = plt.subplots(2, 2, figsize=(11, 7.6))
lim = [0, spk.wer_A.max() * 1.05]
for row, (name, s) in enumerate((('standard', spk), ('forgiven-shared', spk_f))):
    ax = axes[row, 0]
    ax.plot(lim, lim, 'k--', lw=1, label='no change')
    ax.errorbar(s.wer_A, s.wer_B, yerr=[s.wer_B - s.wer_B_lo, s.wer_B_hi - s.wer_B], fmt='none', ecolor=C_GAIN, alpha=0.25, lw=0.8)
    ax.scatter(s.wer_A, s.wer_B, s=18, color=C_GAIN, alpha=0.7, edgecolor='white')
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_xlabel('WER — model A'); ax.set_ylabel('WER — model B'); ax.set_title(f'{name}: every point is one speaker'); ax.legend()
    ax = axes[row, 1]
    ax.hist(s.gain_rel, bins=np.linspace(0, 0.7, 36), color=C_GAIN, alpha=0.85)
    ax.axvline(s[s.reliable].gain_rel.median(), color='k', ls=':', lw=1)
    ax.set_xlabel('relative gain (fraction of error removed)'); ax.set_ylabel('number of speakers'); ax.set_title(f'{name}: median {s[s.reliable].gain_rel.median():.1%}')
plt.tight_layout(); save('forgiven_map'); plt.show()

### 12.1 Does the ranking of speakers change?

In [ ]:
r = spk.reliable
for m in ('wer_A', 'wer_B', 'gain_abs', 'gain_rel'):
    print(f'{m:<9} Spearman between standard and forgiven, reliable speakers: {stats.spearmanr(spk.loc[r, m], spk_f.loc[r, m]).statistic:.3f}')
print(f'speakers hurt by the fine-tune: standard {(spk[r].gain_abs < 0).sum()}, forgiven {(spk_f[r].gain_abs < 0).sum()} of {int(r.sum())}')
panel = pd.read_csv(os.path.join('..', 'src', 'evaluation', 'outputs', 'committees_panel.csv'), index_col=0)
core = panel[~panel.profile.str.contains('alt')]
fig, ax = plt.subplots(figsize=(6.2, 5.2))
ax.scatter(spk.loc[r, 'gain_rel'], spk_f.loc[r, 'gain_rel'], s=14, color='0.7', alpha=0.7, edgecolor='white', label='all reliable speakers')
colors = {'S1': '#D55E00', 'S2': '#CC79A7', 'S3': '#0072B2', 'S4': '#009E73', 'C': '#E69F00'}
for sid, p_ in core.iterrows():
    ax.scatter(spk.loc[sid, 'gain_rel'], spk_f.loc[sid, 'gain_rel'], s=55, color=colors[p_.profile], edgecolor='k', zorder=3)
    ax.annotate(p_.speaker_name, (spk.loc[sid, 'gain_rel'], spk_f.loc[sid, 'gain_rel']), xytext=(5, 3), textcoords='offset points', fontsize=7.5, color=colors[p_.profile])
lim = [0.05, 0.7]; ax.plot(lim, lim, 'k--', lw=1); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('relative gain, standard'); ax.set_ylabel('relative gain, forgiven-shared'); ax.set_title('The panel under both counts'); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc='lower right')
plt.tight_layout(); save('forgiven_panel'); plt.show()

### 12.2 Subgroups and candidates under the new count

In [ ]:
totals_f = kf.groupby('speaker_id').agg(werr_A_total=('werr_A_f', 'sum'), werr_B_total=('werr_B_f', 'sum'))
G_f = spk_f.join(totals_f).join(attrs)
rank_f = E.rank_rules(G_f, n_perm=2000)
verdict = pd.DataFrame({('standard', 'separates gain'): rank.separates_gain, ('standard', 'separates difficulty'): rank.separates_wer,
                        ('forgiven', 'separates gain'): rank_f.separates_gain, ('forgiven', 'separates difficulty'): rank_f.separates_wer,
                        ('eta² gain', 'standard'): rank.eta2_gain_rel.round(3), ('eta² gain', 'forgiven'): rank_f.eta2_gain_rel.round(3)})
display(verdict)
cand_f = E.candidates(spk_f)
bottom = lambda t: set(t.head(40).index)
print(f'bottom-40 candidate list: {len(bottom(cand) & bottom(cand_f))} of 40 speakers the same under both counts')
pct = {k: t[t.reliable].gain_rel.rank(pct=True) for k, t in (('standard', spk), ('forgiven', spk_f))}
tbl = core[['profile', 'speaker_name']].copy()
for k in pct: tbl[f'gain pct {k}'] = [pct[k].get(s, np.nan) for s in core.index]
tbl.round(2)

### 12.3 Reading it

**What moved.** The level, and B's advantage. Under standard counting B removes 24 % of A's
error; once the words both models heard are no longer charged, it removes 37 %. The two models
fail differently — A's errors are more often genuine mishearings, B's more often the protocol's
omissions — so charging both equally for the protocol flatters A.

**What held.** Every reliable speaker is still helped, none hurt. The ranking of speakers barely
moves (Spearman 0.96 on relative gain, 0.99 on absolute gain), 35 of the bottom-40 candidates are
the same, and every subgroup rule keeps its verdict on gain; country of origin picks up a weak
difficulty verdict it did not have. The subgroup effects are small on both counts.

**What to watch.** One panel speaker, דוד ביטן, moves from the 59th to the 75th percentile of
benefit: part of his apparent difficulty is the protocol, and the fine-tune serves him better than
the standard count suggests. Read together with the standard results, the panel stands and his role is
relabelled — hard, and already well served — with the audio gate as the check that could remove him.

**What this does not settle.** Whether a shared insertion is spoken speech or a shared
hallucination. That needs human verbatim text on a small sample.

In [ ]:
# the writer takes the UNFILTERED scored table, so its effect table has all three rows
E.run(scoring='forgiven', seg=X.forgiven_counts(seg), verbose=False)['summary']['corpus']

---
## What this version deliberately leaves out

* **Only one filter.** Quality catches references that do not match their audio. It does not
  catch chunks where the reference is *short* for the audio: after the filter, chunks under
  50 words per minute (1.3 % of chunks, 3 h) still score a WER of 2.0 under both arms — the
  models transcribe speech the protocol condensed, and every word is an insertion. They are a
  small share of the words and are left in, as Stage 1 left in the transcription side.
* **No check of the transcription.** A looping decoder is counted in full; `runaway` flags 306
  chunks for A and 420 for B and is reported, not filtered.
* **No covariate model.** Age, gender, origin and rate are examined one rule at a time.
* **The labels.** Every number above assumes the protocol named the audible speaker. §6 shows
  the filter's footprint tracks WER; the audio gate is the check, and it has not run.
* **The digit problem**, as before: the protocol writes `שמונה`, Whisper writes `8`.